In [ ]:
!pip install gdown

In [ ]:
import polars as pl
import numpy as np
from sentence_transformers import SentenceTransformer
import torch
import gdown
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI
import os

In [ ]:
# File ID from your link
file_id = "1_ntEIwHGuNq3WYzi82uoT1hCN2XtI3Bm"
url = f"https://drive.google.com/uc?id={file_id}"

# Download and save as CSV
output = "movies_df.parquet"
gdown.download(url, output, quiet=False)


Downloading...
From: https://drive.google.com/uc?id=1_ntEIwHGuNq3WYzi82uoT1hCN2XtI3Bm
To: /content/movies_df.parquet
100%|██████████| 46.8M/46.8M [00:00<00:00, 68.0MB/s]


'movies_df.parquet'

In [ ]:
df = pl.read_parquet('/content/movies_df.parquet')
df.head(3)


id,title,vote_average,vote_count,status,release_date,revenue,runtime,budget,imdb_id,original_language,original_title,overview,popularity,tagline,genres,production_companies,production_countries,spoken_languages,cast,director,director_of_photography,writers,producers,music_composer,imdb_rating,imdb_votes,poster_path,keywords,adult,input,movie_link,poster_url,release_year
i64,str,f64,f64,str,str,f64,f64,f64,str,str,str,str,f64,str,str,str,str,str,str,str,str,str,str,str,f64,f64,str,str,bool,str,str,str,i32
313784,"""Carike En Ghoempie In Kinderla…",0.0,0.0,"""Released""","""2007-01-01""",0.0,44.0,0.0,null,"""en""","""Carike En Ghoempie In Kinderla…","""Carike Keuzenkamp in this Afri…",0.6,null,"""Music""",null,null,"""English, Afrikaans""","""""","""""",null,"""""",null,null,0.0,0.0,null,"""afrikaans, ghoempie, kinderlan…",false,"""Genres: Music. Directors: . Wr…","""https://www.themoviedb.org/mov…","""https://via.placeholder.com/50…",2007
29464,"""Uptown Saturday Night""",6.1,38.0,"""Released""","""1974-07-26""",6.7e6,104.0,3e6,"""tt0072351""","""en""","""Uptown Saturday Night""","""Two blue-collar buddies search…",0.4578,"""They get funny when you mess w…","""Comedy, Crime, Action""","""Verdon Productions Limited, Fi…","""United States of America""","""English, Español""","""Lee Chamberlin, Gene McDaniels…","""Sidney Poitier""","""Fred J. Koenekamp""","""Richard Wesley""","""Melville Tucker""","""Tom Scott""",6.6,3059.0,"""/z5dpyCbIDFYUuooLnSAXYNCWWPu.j…","""robbery, gambling, ghetto, mal…",false,"""Genres: Comedy, Crime, Action.…","""https://www.imdb.com/title/tt0…","""https://image.tmdb.org/t/p/w50…",1974
199659,"""Viral Assassins""",3.0,3.0,"""Released""","""2000-10-05""",0.0,90.0,0.0,"""tt0405452""","""en""","""Viral Assassins""","""To some, it’s murder. To them,…",0.0956,"""Violence is contagious!""","""Science Fiction, Drama""","""Troma Entertainment, Shush Lad…","""United States of America""","""English""","""Paul Larkin, Sherri Hewell, St…","""Robert Larkin""",null,"""Robert Larkin""",null,null,4.8,81.0,"""/y0S8uJfZvEeoxagBSH3tGJD2MJv.j…","""government conspiracy, virus, …",false,"""Genres: Science Fiction, Drama…","""https://www.imdb.com/title/tt0…","""https://image.tmdb.org/t/p/w50…",2000


In [ ]:
df.shape

(72462, 34)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# Stop the current cell first!
model =  SentenceTransformer("BAAI/bge-base-en-v1.5", device=device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
df = df.with_columns([
    pl.concat_str([
        pl.lit("Title: "), pl.col("title").fill_null(""),
        pl.lit(". Genres: "), pl.col("genres").fill_null(""),
        pl.lit(". Directors: "), pl.col("director").fill_null(""),
        pl.lit(". Plot: "), pl.col("overview").fill_null(""),
        pl.lit(". Keywords: "), pl.col("keywords").fill_null("")
    ]).alias("input")
])

In [ ]:
# Select the column, then slice the first 8 rows
df.select("input").row(0)[0]


'Title: Carike En Ghoempie In Kinderland 4. Genres: Music. Directors: . Plot: Carike Keuzenkamp in this Afrikaans sing-along DVD for children.. Keywords: afrikaans, ghoempie, kinderland'

In [ ]:
input = df['input'].to_list()

In [ ]:
movie_embeddings = model.encode(
    sentences=input,
    batch_size=256,                 # increase if GPU memory allows
    show_progress_bar=True,
    convert_to_numpy=True,
    device=device,                   # force GPU if available
    normalize_embeddings=True
)


Batches:   0%|          | 0/284 [00:00<?, ?it/s]

In [ ]:
movie_embeddings.shape

(72462, 768)

In [ ]:
df_embeddings = df.with_columns([
    pl.Series(name="embeddings", values=movie_embeddings.tolist())
])

df_embeddings.write_parquet('movies_with_embeddings.parquet')

# 5. Download this file to your computer
from google.colab import files
files.download('movies_with_embeddings.parquet')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df_embeddings.head(3)

id,title,vote_average,vote_count,status,release_date,revenue,runtime,budget,imdb_id,original_language,original_title,overview,popularity,tagline,genres,production_companies,production_countries,spoken_languages,cast,director,director_of_photography,writers,producers,music_composer,imdb_rating,imdb_votes,poster_path,keywords,adult,input,movie_link,poster_url,release_year,embeddings
i64,str,f64,f64,str,str,f64,f64,f64,str,str,str,str,f64,str,str,str,str,str,str,str,str,str,str,str,f64,f64,str,str,bool,str,str,str,i32,list[f64]
313784,"""Carike En Ghoempie In Kinderla…",0.0,0.0,"""Released""","""2007-01-01""",0.0,44.0,0.0,null,"""en""","""Carike En Ghoempie In Kinderla…","""Carike Keuzenkamp in this Afri…",0.6,null,"""Music""",null,null,"""English, Afrikaans""","""""","""""",null,"""""",null,null,0.0,0.0,null,"""afrikaans, ghoempie, kinderlan…",false,"""Title: Carike En Ghoempie In K…","""https://www.themoviedb.org/mov…","""https://via.placeholder.com/50…",2007,"[-0.059526, 0.009719, … 0.006406]"
29464,"""Uptown Saturday Night""",6.1,38.0,"""Released""","""1974-07-26""",6.7e6,104.0,3e6,"""tt0072351""","""en""","""Uptown Saturday Night""","""Two blue-collar buddies search…",0.4578,"""They get funny when you mess w…","""Comedy, Crime, Action""","""Verdon Productions Limited, Fi…","""United States of America""","""English, Español""","""Lee Chamberlin, Gene McDaniels…","""Sidney Poitier""","""Fred J. Koenekamp""","""Richard Wesley""","""Melville Tucker""","""Tom Scott""",6.6,3059.0,"""/z5dpyCbIDFYUuooLnSAXYNCWWPu.j…","""robbery, gambling, ghetto, mal…",false,"""Title: Uptown Saturday Night. …","""https://www.imdb.com/title/tt0…","""https://image.tmdb.org/t/p/w50…",1974,"[-0.072058, -0.015014, … -0.013055]"
199659,"""Viral Assassins""",3.0,3.0,"""Released""","""2000-10-05""",0.0,90.0,0.0,"""tt0405452""","""en""","""Viral Assassins""","""To some, it’s murder. To them,…",0.0956,"""Violence is contagious!""","""Science Fiction, Drama""","""Troma Entertainment, Shush Lad…","""United States of America""","""English""","""Paul Larkin, Sherri Hewell, St…","""Robert Larkin""",null,"""Robert Larkin""",null,null,4.8,81.0,"""/y0S8uJfZvEeoxagBSH3tGJD2MJv.j…","""government conspiracy, virus, …",false,"""Title: Viral Assassins. Genres…","""https://www.imdb.com/title/tt0…","""https://image.tmdb.org/t/p/w50…",2000,"[0.02644, -0.001107, … 0.021951]"


In [ ]:
def verify_embeddings(num_samples=3):
    # Pick random rows to test
    samples = df_embeddings.sample(num_samples)

    for row in samples.to_dicts():
        title = row['title']
        text_to_test = row['input']
        stored_vector = np.array(row['embeddings']).reshape(1, -1)

        # Re-encode the text fresh
        print(f"Verifying: {title}...")
        fresh_vector = model.encode(text_to_test).reshape(1, -1)

        # Calculate Cosine Similarity
        score = cosine_similarity(stored_vector, fresh_vector)[0][0]

        print(f"Similarity Score: {score:.4f}")
        if score > 0.99:
            print("Match: This embedding matches the text perfectly.")
        else:
            print("Mismatch: This embedding does NOT match this text!")
        print("-" * 30)

verify_embeddings(3)

Verifying: Rubberbandits Guide to 1916...
Similarity Score: 1.0000
Match: This embedding matches the text perfectly.
------------------------------
Verifying: The Twilight People...
Similarity Score: 1.0000
Match: This embedding matches the text perfectly.
------------------------------
Verifying: Mary Goes Round...
Similarity Score: 1.0000
Match: This embedding matches the text perfectly.
------------------------------


In [ ]:
def search_by_query(query: str, df: pl.DataFrame, embeddings: np.ndarray, model, top_n: int = 10):
    # 1. ADD THE INSTRUCTION PREFIX (Required for BGE v1.5)
    instruction = "Represent this sentence for searching relevant passages: "
    full_query = instruction + query

    # 2. Encode AND Normalize the query
    # BGE works best when you explicitly tell it it's a query
    query_vector = model.encode(full_query, normalize_embeddings=True).reshape(1, -1)

    # 3. Calculate similarities
    similarities = (query_vector @ embeddings.T).flatten()

    # 4. Get the top indices
    top_indices = np.argsort(-similarities)[:top_n]

    return df[top_indices, ["title", "overview", "genres", "director", "keywords"]]


# --- EXECUTION ---
query = "drama thriller movie about two magicians who became competitors and used Tesla coin, directed by Nolan"
results = search_by_query(query, df_embeddings, movie_embeddings, model)
print(results)


shape: (10, 5)
┌───────────────────┬───────────────────┬───────────────────┬───────────────────┬──────────────────┐
│ title             ┆ overview          ┆ genres            ┆ director          ┆ keywords         │
│ ---               ┆ ---               ┆ ---               ┆ ---               ┆ ---              │
│ str               ┆ str               ┆ str               ┆ str               ┆ str              │
╞═══════════════════╪═══════════════════╪═══════════════════╪═══════════════════╪══════════════════╡
│ The Prestige      ┆ A mysterious      ┆ Drama, Mystery,   ┆ Christopher Nolan ┆ dying and death, │
│                   ┆ story of two      ┆ Science Fictio…   ┆                   ┆ suicide, clas…   │
│                   ┆ magi…             ┆                   ┆                   ┆                  │
│ The Magician      ┆ A traveling       ┆ Drama             ┆ Ingmar Bergman    ┆ dual identity,   │
│                   ┆ magician and his  ┆                   ┆               

In [ ]:
client = OpenAI(api_key=os.getenv("GROQ_API_KEY"),  base_url="https://api.groq.com/openai/v1")

def generate_recommendation(user_query: str, search_results: pl.DataFrame):
    context = ""
    # Use iter_rows for clean Polars iteration
    for row in search_results.iter_rows(named=True):
        context += f"Title: {row['title']}\n"
        context += f"Genres: {row['genres']}\n"
        context += f"Director: {row['director']}\n"
        context += f"Plot: {row['overview']}\n"
        context += f"Keywords: {row['keywords']}\n"


    system_prompt = """
    You are a professional movie critic and recommendation engine.
    Use the provided movie context to answer the user's request.
    Explain WHY you are recommending these specific movies based on their plot and cast.
    If the context doesn't contain a relevant movie, say you don't know.
    """

    user_prompt = f"User Request: {user_query}\n\nMovie Context:\n{context}"

    # CRITICAL: Use a real Groq model name to avoid 404 errors
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.7
    )

    return response.choices[0].message.content


answer = generate_recommendation(query, results)

print("AI RECOMMENDATION:\n", answer)

AI RECOMMENDATION:
 **Recommendation: *The Prestige* (2006)**  

**Why this film fits your request perfectly**

| Criterion | How *The Prestige* Meets It |
|-----------|-----------------------------|
| **Genre** | It’s a blend of drama, mystery, and thriller, with a tense, psychological edge that keeps you guessing. |
| **Plot** | Centers on two stage magicians—Robert Angier (Hugh Jackman) and Alfred Borden (Christian Bale)—whose intense rivalry drives them to increasingly dangerous extremes. The story is a classic “who’s doing what” cat‑and‑mouse game, perfect for a drama‑thriller vibe. |
| **Tesla Coil / Tesla Coin** | The film famously incorporates a real Tesla coil as a pivotal plot device. It’s the “Tesla” element you mentioned, and it’s woven into the narrative as a symbol of obsession and the peril of playing with unseen forces. |
| **Director** | Christopher Nolan, renowned for his mind‑bending storytelling and meticulous pacing—exactly what you’re looking for. |
| **Cast** | C